# scVI / scANVI Annotation — Cisplatin_GC_Vehicle (4 conditions)

**Author**: Xin Wang  
**Date**: 2026-04-14  

This notebook runs scVI/scANVI label transfer onto the 4-condition query dataset  
(Vehicle, Vehicle-GC, Cisplatin, Cisplatin-GC) using **two reference atlases**:

| Reference | File |
|-----------|------|
| Mouse Kidney Atlas (MKA) | `MouseKidneyATLAS_MKA_updated.h5ad` |
| Lake 2025 snRNA-seq | `mouse_kidney_snRNAseq_Lake2025_bioRxiv_V2.h5ad` |

**Pipeline:**
1. Load both reference h5ads; harmonise gene names and cell-type labels  
2. Load query h5ad (`Cisplatin_GC_Vehicle_Raw_Predict_Annotation.h5ad`)  
3. Merge references (batch-aware) and subset to shared genes with query  
4. Train scVI on merged reference  
5. Merge reference + query; train scANVI (semi-supervised)  
6. Predict cell types on query cells and save CSV  

**Prerequisites**: Run `Cisplatin_GC_Vehicle_Annotation.Rmd` first to generate the query h5ad.

In [9]:
# Install dependencies if needed
# !pip install scanpy scvi-tools mygene

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import matplotlib.pyplot as plt
import torch

scvi.settings.seed = 0
sc.settings.verbosity = 1
print("scvi-tools version:", scvi.__version__)
print("GPU available:", torch.cuda.is_available())

/home/gdbecknelllab/xxw004/Workspace/.conda/envs/cell2loc_env/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/gdbecknelllab/xxw004/Workspace/.conda/envs/cell2loc_env/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/gdbecknelllab/xxw004/Workspace/.conda/envs/cell2loc_env/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/gdbecknelllab/xxw004/Workspace/.conda/envs/cell2loc_env/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing CSCDataset from `anndata.experimental` is deprecated. Import anndata.abc.CSCDataset instead.
  warnings.warn(ms

scvi-tools version: 1.2.1
GPU available: False


## Paths

In [4]:
ref_dir   = "/home/gdzepedaorozcolab/lab/xxw004/Projects/RawDZscRNAseq/Datasets/Reference/"
ann_dir   = "/home/gdzepedaorozcolab/lab/xxw004/Projects/RawDZscRNAseq/Results/Integration/IntegrationGC/Annotation/"

mka_path   = os.path.join(ref_dir, "MouseKidneyATLAS_MKA_updated.h5ad")
lake_path  = os.path.join(ref_dir, "mouse_kidney_snRNAseq_Lake2025_bioRxiv_V2.h5ad")
query_path = os.path.join(ann_dir, "Cisplatin_GC_Vehicle_Raw_Predict_MKA_Lake_Annotation.h5ad")
out_dir    = ann_dir
os.makedirs(out_dir, exist_ok=True)

for p in [mka_path, lake_path, query_path]:
    print(f"{os.path.basename(p):60s}  exists={os.path.exists(p)}")

MouseKidneyATLAS_MKA_updated.h5ad                             exists=True
mouse_kidney_snRNAseq_Lake2025_bioRxiv_V2.h5ad                exists=True
Cisplatin_GC_Vehicle_Raw_Predict_MKA_Lake_Annotation.h5ad     exists=True


## Load and inspect both reference datasets

In [5]:
adata_mka  = sc.read_h5ad(mka_path)
adata_lake = sc.read_h5ad(lake_path)

print("=== MKA reference ===")
print(adata_mka)
print(adata_mka.obs.columns.tolist())

print("\n=== Lake reference ===")
print(adata_lake)
print(adata_lake.obs.columns.tolist())

=== MKA reference ===
AnnData object with n_obs × n_vars = 141401 × 16641
    obs: 'Origin', 'suspension_type', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'author_cell_type', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'development_stage_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'donor_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'schema_reference

In [7]:
# -----------------------------------------------------------------------
# Identify the cell-type column in each reference.
# Update the values below if the column names differ in your h5ads.
# -----------------------------------------------------------------------
mka_label_col  = "author_cell_type"   # <-- update if needed
lake_label_col = "SubclassLevel1"   # <-- update if needed

print("MKA cell types:")
print(adata_mka.obs[mka_label_col].value_counts())

print("\nLake cell types:")
print(adata_lake.obs[lake_label_col].value_counts())

MKA cell types:

Lake cell types:
SubclassLevel1
PT          139750
TAL          54327
PC           23545
DCT          23071
EC           20260
IC           11551
DTL          11112
FIB          10254
Myeloid       8769
Lymphoid      3104
CNT           2672
PapE          2009
POD           1715
ATL           1677
PEC           1003
VSM/P          849
Ad              69
NEU             27
Name: count, dtype: int64


## Harmonise gene names (Ensembl → symbol if needed)

In [ ]:
# Inspect var_names and feature_name to confirm format before conversion
print("MKA var_names[:5]   :", adata_mka.var_names[:5].tolist())
print("MKA feature_name[:5]:", adata_mka.var["feature_name"][:5].tolist())
print()
print("Lake var_names[:5]   :", adata_lake.var_names[:5].tolist())
print("Lake feature_name[:5]:", adata_lake.var["feature_name"][:5].tolist())

In [ ]:
import re

def ensembl_to_symbol(adata):
    """Convert Ensembl IDs in var_names to clean gene symbols using adata.var['feature_name']."""
    if not adata.var_names[0].startswith("ENSMUS"):
        print("var_names already look like gene symbols — skipping conversion.")
        return adata
    if "feature_name" not in adata.var.columns:
        raise ValueError("No 'feature_name' column found in adata.var.")
    symbols = adata.var["feature_name"].astype(str).values.copy()

    # Strip trailing _ENSMUSG... suffix (e.g. "Serpinb8_ENSMUSG00000026315" → "Serpinb8")
    symbols = np.array([re.sub(r'_ENSMUSG\d+$', '', s) for s in symbols])

    # Fall back to Ensembl ID wherever feature_name is missing/nan
    fallback_mask = (symbols == "nan") | (symbols == "") | (symbols == "None")
    n_fallback = fallback_mask.sum()
    if n_fallback:
        print(f"  WARNING: {n_fallback} genes have no feature_name — keeping Ensembl ID as fallback.")
        symbols[fallback_mask] = adata.var_names[fallback_mask]

    adata.var_names = symbols
    adata.var_names_make_unique()
    print(f"  Converted {adata.n_vars} var_names to gene symbols ({n_fallback} kept as Ensembl IDs).")
    return adata

adata_mka  = ensembl_to_symbol(adata_mka)
adata_lake = ensembl_to_symbol(adata_lake)
print("MKA var sample:",  adata_mka.var_names[:5].tolist())
print("Lake var sample:", adata_lake.var_names[:5].tolist())

# ── Sanity check: make sure MKA cell-type labels are populated ──────────────
n_mka_labeled = adata_mka.obs[mka_label_col].notna().sum()
print(f"\nMKA obs with '{mka_label_col}' populated: {n_mka_labeled} / {adata_mka.n_obs}")
if n_mka_labeled == 0:
    raise ValueError(
        f"Column '{mka_label_col}' is entirely empty in adata_mka. "
        "Check the correct column name — run: print(adata_mka.obs.columns.tolist())"
    )

## Load query dataset (4-condition)

In [ ]:
adata_query = sc.read_h5ad(query_path)
print(adata_query)
print('Conditions:', adata_query.obs['DataSet'].value_counts().to_dict())

## Annotate query against each reference independently

Each reference (MKA, Lake) is used separately:
1. Find shared genes between reference and query
2. Train scVI on the reference alone
3. Merge reference + query and train scANVI
4. Predict labels on query cells
5. Save per-reference CSV

In [ ]:
LABEL_KEY = "cell_type"
all_preds = {}   # will hold one DataFrame per reference

for ref_name, adata_ref, label_col in [
    ('MKA',  adata_mka,  mka_label_col),
    ('Lake', adata_lake, lake_label_col),
]:
    print(f"\n{'='*60}")
    print(f"  {ref_name} reference")
    print(f"{'='*60}")

    # ── unified label column ────────────────────────────────────────
    if label_col != LABEL_KEY:
        adata_ref.obs[LABEL_KEY] = adata_ref.obs[label_col]

    # ── shared genes ────────────────────────────────────────────────
    shared = adata_ref.var_names.intersection(adata_query.var_names)
    print(f"  Shared genes ({ref_name} ∩ query): {len(shared)}")

    ref_sub   = adata_ref[:, shared].copy()
    query_sub = adata_query[:, shared].copy()

    # ── scVI on reference ───────────────────────────────────────────
    scvi.model.SCVI.setup_anndata(ref_sub, labels_key=LABEL_KEY)
    scvi_model = scvi.model.SCVI(ref_sub)
    scvi_model.train()
    scvi_model.save(os.path.join(out_dir, f"scvi_model_{ref_name}"), overwrite=True)
    print(f"  scVI model saved.")

    # reference UMAP
    ref_sub.obsm['X_scVI'] = scvi_model.get_latent_representation()
    sc.pp.neighbors(ref_sub, use_rep='X_scVI')
    sc.tl.umap(ref_sub)
    fig, ax = plt.subplots(figsize=(9, 6))
    sc.pl.umap(ref_sub, color=LABEL_KEY, title=f'{ref_name} reference — cell type', ax=ax, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{ref_name}_scVI_UMAP.pdf'), bbox_inches='tight')
    plt.show()

    # ── prepare query for scANVI ────────────────────────────────────
    scvi.model.SCVI.prepare_query_anndata(query_sub, scvi_model)
    query_sub.obs[LABEL_KEY] = pd.Categorical(['Unknown'] * query_sub.n_obs)

    # ── merge ref + query ───────────────────────────────────────────
    combined = ref_sub.concatenate(
        query_sub,
        batch_key='batch',
        batch_categories=['ref', 'query']
    )

    # ── scANVI ──────────────────────────────────────────────────────
    scvi.model.SCANVI.setup_anndata(
        combined,
        labels_key=LABEL_KEY,
        batch_key='batch',
        unlabeled_category='Unknown'
    )
    scanvi_model = scvi.model.SCANVI(combined)
    scanvi_model.train(max_epochs=50)
    scanvi_model.save(os.path.join(out_dir, f"scanvi_model_{ref_name}"), overwrite=True)
    print(f"  scANVI model saved.")

    # ── predict on query ────────────────────────────────────────────
    query_mask  = combined.obs['batch'] == 'query'
    soft_preds  = scanvi_model.predict(combined[query_mask], soft=True)
    label_names = scanvi_model.adata_manager.get_state_registry('labels')['categorical_mapping']

    prob_df     = pd.DataFrame(soft_preds, index=combined.obs_names[query_mask], columns=label_names)
    pred_labels = prob_df.idxmax(axis=1)
    pred_conf   = prob_df.max(axis=1)

    combined.obs.loc[query_mask, f'scanvi_{ref_name}_label']      = pred_labels.values
    combined.obs.loc[query_mask, f'scanvi_{ref_name}_confidence'] = pred_conf.values

    # confidence histogram
    pred_conf.hist(bins=50, figsize=(7, 4))
    plt.title(f'scANVI confidence — {ref_name} reference')
    plt.xlabel('Confidence'); plt.ylabel('Cell count')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'scanvi_{ref_name}_confidence.pdf'))
    plt.show()

    # combined UMAP
    combined.obsm['X_scANVI'] = scanvi_model.get_latent_representation()
    sc.pp.neighbors(combined, use_rep='X_scANVI')
    sc.tl.umap(combined)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sc.pl.umap(combined, color=LABEL_KEY,             title=f'{ref_name} — cell type',  ax=axes[0], show=False)
    sc.pl.umap(combined, color='batch',               title=f'{ref_name} — ref / query', ax=axes[1], show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{ref_name}_combined_scANVI_UMAP.pdf'), bbox_inches='tight')
    plt.show()

    # store predictions
    all_preds[ref_name] = combined.obs.loc[
        query_mask,
        [f'scanvi_{ref_name}_label', f'scanvi_{ref_name}_confidence']
    ].copy()
    print(f"  {ref_name} done.\n")

In [ ]:
# Merge both per-reference predictions and save combined CSV
preds_combined = all_preds['MKA'].join(all_preds['Lake'], how='outer')

# Append any existing Seurat/manual columns from the query obs if present
extra_cols = [c for c in adata_query.obs.columns
              if c in ('predicted.id', 'prediction.score.max', 'Raw_cell_type', 'Raw_cell_type_Confidence')]
if extra_cols:
    preds_combined = preds_combined.join(adata_query.obs[extra_cols], how='left')

csv_path = os.path.join(out_dir, "Cisplatin_GC_Vehicle_scanvi_annotations.csv")
preds_combined.to_csv(csv_path)
print("Combined predictions saved to:", csv_path)
preds_combined.head()